# ❄️ Permafrost Framework — Quick Start

Aprenda a usar o Permafrost em 5 minutos.

[![PyPI version](https://badge.fury.io/py/permafrost-framework.svg)](https://pypi.org/project/permafrost-framework/)

**O que vamos fazer:**
1. Instalar o Permafrost
2. Comprimir um DataFrame (`freeze`)
3. Restaurar sem perda (`thaw`)
4. Inspecionar sem descomprimir (`audit`)
5. Ver a economia de espaço

In [1]:
# Instalar (rodar apenas uma vez)
%pip install permafrost-framework --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import permafrost as pf
import pandas as pd
import numpy as np
import tempfile, os

print(f"Permafrost {pf.__version__} carregado")

permafrost_schema_detector.py OK
  Classes: SchemaDetector, DataType, FieldKind


permafrost.spark OK
  Classes: PermafrostDataSource, _PermafrostReader, _PermafrostWriter
  Funções: register(spark)
Permafrost 0.6.3 carregado


In [3]:
import tempfile, os
WORKDIR = tempfile.mkdtemp(prefix='pf_demo_')
print(f'Arquivos temporários em: {WORKDIR}')

Arquivos temporários em: C:\Users\CAUFER~1\AppData\Local\Temp\pf_demo_i519qr8b


In [4]:
import tempfile, os
WORKDIR = tempfile.mkdtemp(prefix="pf_demo_")
print(f"Arquivos temporários em: {WORKDIR}")

Arquivos temporários em: C:\Users\CAUFER~1\AppData\Local\Temp\pf_demo_cx0esc9w


## 1. Criando dados de exemplo

Simularemos um dataset de vendas corporativas com diferentes tipos de coluna:
timestamps, categorias, inteiros e floats.

In [5]:
np.random.seed(42)
N = 80_000

df = pd.DataFrame({
    'timestamp':  pd.date_range('2020-01-01', periods=N, freq='1min'),
    'produto_id': np.random.randint(1, 10_000, N),
    'categoria':  np.random.choice(['Eletronicos','Vestuario','Alimentos','Moveis','Esportes'], N),
    'regiao':     np.random.choice(['Sul','Norte','Leste','Oeste','Centro'], N),
    'quantidade': np.random.randint(1, 50, N),
    'preco':      np.round(np.random.uniform(10.0, 5000.0, N), 2),
    'desconto':   np.round(np.random.uniform(0.0, 0.5, N), 3),
    'aprovado':   np.random.choice([True, False], N),
    'loja_id':    np.random.randint(1, 500, N),
})

print(f"Dataset: {len(df):,} linhas × {len(df.columns)} colunas")
df.dtypes

Dataset: 80,000 linhas × 9 colunas


timestamp     datetime64[us]
produto_id             int32
categoria                str
regiao                   str
quantidade             int32
preco                float64
desconto             float64
aprovado                bool
loja_id                int32
dtype: object

## 2. Freeze — comprimindo

O `freeze()` aplica preditores colunares antes do codec LZMA2:
- `ts_delta_s`: timestamps → deltas zigzag
- `delta_zigzag`: inteiros → deltas
- `category_u8`: strings categóricas → índice 8-bit
- `raw_text`: strings livres

In [6]:
with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, 'vendas.permafrost')
    csv_path = os.path.join(tmpdir, 'vendas.csv')

    # Salvar CSV para comparação
    df.to_csv(csv_path, index=False)
    csv_mb = os.path.getsize(csv_path) / 1e6

    # Freeze com LZMA2 (máxima compressão)
    metrics = pf.freeze(df, path, codec=pf.CODEC_LZMA2)

    print(f"\n{'='*50}")
    print(f"CSV bruto:         {csv_mb:.2f} MB")
    print(f".permafrost:       {metrics['stored_mb']:.2f} MB")
    print(f"Ratio:             {metrics['ratio']:.2f}×")
    print(f"Redução:           {metrics['reduction_pct']:.1f}%")
    print(f"Tempo:             {metrics['freeze_s']:.2f}s")

    # Guardar o path para as próximas células
    import shutil
    final_path = os.path.join(WORKDIR, 'vendas_demo.permafrost')
    shutil.copy(path, final_path)

print(f"\nArquivo salvo em: {final_path}")


CSV bruto:         5.36 MB
.permafrost:       0.71 MB
Ratio:             7.50×
Redução:           86.7%
Tempo:             1.25s

Arquivo salvo em: C:\Users\CAUFER~1\AppData\Local\Temp\pf_demo_cx0esc9w\vendas_demo.permafrost


## 3. Thaw — restaurando

O `thaw()` restaura o DataFrame original com verificação de integridade SHA-256.

In [7]:
df_back = pf.thaw(final_path, verify=True)

print(f"Linhas restauradas: {len(df_back):,}")
print(f"Colunas: {list(df_back.columns)}")

# Verificação de fidelidade
for col in df.columns:
    if df[col].dtype == 'bool':
        ok = df[col].equals(df_back[col])
    elif pd.api.types.is_float_dtype(df[col]):
        ok = np.allclose(df[col].values, df_back[col].values, rtol=1e-5)
    else:
        ok = df[col].equals(df_back[col])
    status = '✓' if ok else '✗'
    print(f"  {status} {col}")

Linhas restauradas: 80,000
Colunas: ['timestamp', 'produto_id', 'categoria', 'regiao', 'quantidade', 'preco', 'desconto', 'aprovado', 'loja_id']
  ✗ timestamp
  ✗ produto_id
  ✗ categoria
  ✗ regiao
  ✗ quantidade
  ✓ preco
  ✗ desconto
  ✗ aprovado
  ✗ loja_id


## 4. Audit — inspecionando sem descomprimir

O `audit()` lê apenas o header e o sparse index — sem descomprimir os dados.

In [8]:
info = pf.audit(final_path)

print("Informações do arquivo:")
for k, v in info.items():
    if k not in ('columns', 'index_entries', 'manifests'):
        print(f"  {k:20s}: {v}")

print(f"\n  {'columns':20s}: {info['columns']}")

Informações do arquivo:
  version             : 1.3
  codec               : lzma2
  quant               : 0
  freeze_date         : 2026-05-15T13:20:23
  orig_rows           : 80000
  n_chunks            : 8
  chunk_rows          : 10000
  file_size_mb        : 0.714
  partition_col       : __rows__
  partition_keys      : ['rows_0_9999', 'rows_10000_19999', 'rows_20000_29999', 'rows_30000_39999', 'rows_40000_49999', 'rows_50000_59999', 'rows_60000_69999', 'rows_70000_79999']
  comment             : 

  columns             : ['timestamp', 'produto_id', 'categoria', 'regiao', 'quantidade', 'preco', 'desconto', 'aprovado', 'loja_id']


## 5. Particionamento — leitura seletiva

Com `partition_by`, o Permafrost cria um sparse index que permite ler
apenas as linhas de uma categoria sem descomprimir o restante.

In [9]:
path_part = '' + WORKDIR + '/vendas_particionado.permafrost'

metrics_part = pf.freeze(df, path_part,
                         codec=pf.CODEC_LZMA2,
                         partition_by='regiao')

print(f"Chunks criados: {metrics_part['n_chunks']}")
print(f"Índice sparse:  {metrics_part['index_entries']} entradas")

# Leitura seletiva — só a região Sul
df_sul = pf.thaw(path_part, filter={'regiao': 'Sul'})
print(f"\nLinhas da região Sul: {len(df_sul):,}")
print(f"Regiões únicas: {df_sul['regiao'].unique().tolist()}")

Chunks criados: 8
Índice sparse:  8 entradas

Linhas da região Sul: 15,963
Regiões únicas: ['Sul']


## 6. Comparativo de codecs

Permafrost suporta três codecs com diferentes trade-offs entre velocidade e ratio.

In [10]:
import time

codecs = [
    ('ZSTD (rápido)',  pf.CODEC_ZSTD),
    ('LZMA2 (padrão)', pf.CODEC_LZMA2),
]

results = []
for name, codec in codecs:
    p = f'' + WORKDIR + '/bench_{codec}.permafrost'
    t0 = time.time()
    m = pf.freeze(df, p, codec=codec)
    results.append({
        'Codec':    name,
        'Tamanho':  f"{m['stored_mb']:.2f} MB",
        'Ratio':    f"{m['ratio']:.2f}×",
        'Tempo':    f"{m['freeze_s']:.2f}s",
    })

pd.DataFrame(results).set_index('Codec')

,Tamanho,Ratio,Tempo
Codec,,,
ZSTD (rápido),0.83 MB,6.42×,0.90s
LZMA2 (padrão),0.71 MB,7.50×,1.18s


## Resumo

| Operação | Comando | Tempo típico |
|----------|---------|-------------|
| Comprimir | `pf.freeze(df, path)` | ~2–5s para 80k linhas |
| Restaurar | `pf.thaw(path)` | <1s |
| Inspecionar | `pf.audit(path)` | <50ms |
| Leitura seletiva | `pf.thaw(path, filter={col: val})` | <1s |

**Próximos passos:**
- `02_social_media_jsonl.ipynb` — dados JSONL / NoSQL
- `03_mongodb_dump.ipynb` — documentos aninhados
- `04_s3_glacier_archive.ipynb` — cloud + lifecycle
- `05_cluster_docker.ipynb` — processamento distribuído